# MindStream — CV Emotion Model (Model 1)
TensorFlow/Keras, MobileNetV2 backbone, FER2013 → 7-class emotion classifier.

Two-phase training: head warmup (backbone frozen) → fine-tune (top layers unfrozen).

In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

## Config
Paths match the `E:\\Core-AI\\projects\\mindstream` layout — edit here if yours differs.

In [ ]:
DATA_DIR = r"E:\Core-AI\DATASETS\FER2013"
CHECKPOINT_DIR = r"E:\Core-AI\MODELS\CV\checkpoints"

FER_CLASSES = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
IMG_SIZE = (128, 128)  # 128x128: best compute/accuracy trade-off when upscaling from 48x48
BATCH_SIZE = 32
NUM_CLASSES = len(FER_CLASSES)

EPOCHS_WARMUP = 10
EPOCHS_FINETUNE = 20

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 1. Data augmentation + pipeline
FER2013 images are 48x48 grayscale. We resize to `IMG_SIZE` and replicate the single channel to 3 so MobileNetV2's ImageNet weights apply. Augmentation only runs during training.

In [ ]:
augmentation_layer = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.10),
        layers.RandomZoom(0.08),
        layers.RandomTranslation(0.05, 0.05),
        layers.RandomBrightness(0.10),
    ],
    name="data_augmentation",
)

def build_dataset(directory, training=True):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        labels="inferred",
        label_mode="int",
        class_names=FER_CLASSES,
        color_mode="grayscale",
        image_size=(48, 48),
        batch_size=BATCH_SIZE,
        shuffle=training,
    )

    def _prep(img, label):
        img = tf.image.grayscale_to_rgb(img)
        img = tf.image.resize(img, IMG_SIZE)
        if training:
            img = augmentation_layer(img, training=True)
        img = preprocess_input(img)
        return img, label

    return ds.map(_prep, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

### Sanity check
Confirms the pipeline loads and shapes are correct before building the model.

In [ ]:
train_dir = os.path.join(DATA_DIR, "train")
val_dir = os.path.join(DATA_DIR, "val")

train_ds = build_dataset(train_dir, training=True)
val_ds = build_dataset(val_dir, training=False)

for imgs, labels in train_ds.take(1):
    print("Batch image shape:", imgs.shape)
    print("Batch label shape:", labels.shape)
    print("Pixel value range:", float(imgs.numpy().min()), "to", float(imgs.numpy().max()))
    print("Sample labels:", labels.numpy()[:8])

## 2. Class weights
FER2013 is heavily imbalanced (e.g. `disgust` has far fewer samples than `happy`). Without this, the model will effectively ignore minority classes.

In [ ]:
def get_class_weights(train_dir):
    y_train = []
    for class_idx, class_name in enumerate(FER_CLASSES):
        folder_path = os.path.join(train_dir, class_name)
        if os.path.exists(folder_path):
            count = len(glob.glob(os.path.join(folder_path, "*.*")))
            y_train.extend([class_idx] * count)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=np.array(y_train)
    )
    return dict(enumerate(weights))

class_weights = get_class_weights(train_dir)
print("Class weights:", class_weights)

## 3. Model architecture
MobileNetV2 backbone (frozen initially) + a wider classifier head with BatchNorm and dropout.

In [ ]:
def build_model(input_shape=(128, 128, 3), num_classes=7):
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name="MindStream_MobileNetV2")
    return model, base_model

model, base_model = build_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), num_classes=NUM_CLASSES)
model.summary()

## 4. Callbacks
Best-checkpoint saving, LR reduction on plateau, and early stopping so we don't hand-tune epoch counts.

In [ ]:
checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_emotion_model.keras")
cb_list = [
    callbacks.ModelCheckpoint(checkpoint_path, monitor="val_accuracy", save_best_only=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1)
]

## Phase 1 — Train classification head (backbone frozen)

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss=losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_WARMUP,
    class_weight=class_weights,
    callbacks=cb_list
)

## Phase 2 — Fine-tune MobileNetV2 base
Unfreezes layers from index 100 onward (roughly the top third of the network) at a much lower learning rate.

In [ ]:
base_model.trainable = True

fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss=losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_WARMUP + EPOCHS_FINETUNE,
    initial_epoch=history_phase1.epoch[-1] + 1,
    class_weight=class_weights,
    callbacks=cb_list
)

print(f"\nTraining complete. Best checkpoint saved to:\n  {checkpoint_path}")

## 5. Evaluation
Classification report + confusion matrix on the held-out val set, using the best saved checkpoint.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best_model = tf.keras.models.load_model(checkpoint_path)

y_true, y_pred = [], []
for imgs, labels in val_ds:
    preds = best_model.predict(imgs, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=FER_CLASSES))
print(confusion_matrix(y_true, y_pred))